# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings

# Suppress SettingWithCopy warnings for demonstration purposes only
warnings.filterwarnings('ignore', category=pd.errors.SettingWithCopyWarning)

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The Croissant schema organizes data into *RecordSets*, each containing *Fields* and *Columns*. All references in this notebook use their `@id` fields, as per best practice.

In [ ]:
def print_record_set_overview(ds):
    print("Available Record Sets:")
    record_sets = ds.record_sets
    for rs in record_sets:
        print(f"  - Record Set: {rs['@id']}")
        print(f"    Name: {rs.get('name', '')}")
        print(f"    Description: {rs.get('description', '')}")
        if 'field' in rs:
            print("    Fields:")
            fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
            for fld in fields:
                print(f"      - Field @id: {fld['@id']} (name: {fld.get('name','')})")
                if 'column' in fld:
                    columns = fld['column'] if isinstance(fld['column'],list) else [fld['column']]
                    for col in columns:
                        print(f"         * Column @id: {col['@id']} (name: {col.get('name','')})")
        print()

# Explore the Record Sets and fields:
print_record_set_overview(dataset)

### Show a preview of records for each Record Set

You can use each Record Set's `@id` when extracting records. Below, we'll print the first two records for each Record Set.

In [ ]:
# Show some example records for illustration
for rs in dataset.record_sets:
    rs_id = rs['@id']
    print(f'First records from Record Set {rs_id}:')
    for i, record in enumerate(dataset.records(record_set=rs_id)):
        print(f'  Record #{i+1}:')
        print(record)
        if i >= 1:
            break
    print('-'*50)

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

*If there is only one primary tabular record set, it's usually the one with the clinical and molecular data described in the metadata.*

In [ ]:
# Collect list of available record set @ids (you may modify to focus on a subset if desired)
record_sets_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

# Extract data from each record set using its @id
for record_set_id in record_sets_ids:
    records_list = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records_list)
    dataframes[record_set_id] = df
    print(f"Loaded record set {record_set_id}: {df.shape[0]} records, {df.shape[1]} columns.")

# Show columns and a preview for the first (or main) record set
if record_sets_ids:
    sample_record_set_id = record_sets_ids[0]
    print(f"\nFields (columns) for {sample_record_set_id}:")
    print(dataframes[sample_record_set_id].columns.tolist())
    dataframes[sample_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

*Below, we select a numeric field to filter on. Replace `<numeric_field_id>` and `<group_field_id>` with actual field (column) names or `@id`s as observed above. If there are no numeric fields, explore categorical or textual fields instead.*

In [ ]:
# Find the first record set with at least one numeric field
main_rs = None
numeric_field = None
group_field = None

for rs_id, df in dataframes.items():
    if df.empty:
        continue
    # Detect numeric columns
    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    object_cols = df.select_dtypes(include='object').columns.tolist()
    if numeric_cols:
        main_rs = rs_id
        numeric_field = numeric_cols[0]
        # Try to pick a non-numeric for groupby if available
        group_field = object_cols[0] if object_cols else None
        break

if main_rs is None or numeric_field is None:
    raise RuntimeError('No numeric fields found for EDA! Please review your record set columns.')

print(f"Performing EDA on Record Set: {main_rs}")
print(f"Using numeric field: {numeric_field}")
if group_field:
    print(f"Using group-by field: {group_field}")

# Example: Filter records with numeric_field > threshold
threshold = dataframes[main_rs][numeric_field].quantile(0.75)  # Use upper quartile as example
filtered_df = dataframes[main_rs][dataframes[main_rs][numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold:.2f} (upper quartile): {filtered_df.shape[0]} found.")
print(filtered_df[[numeric_field]].head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (
    (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
)
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# If group_field is present, show group-level statistics
if group_field and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame(name=f"mean_{numeric_field}")
    print(f"\nGrouped mean {numeric_field} by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

*Below, we plot a histogram of the numeric field and, if grouping is possible, a boxplot by group.*

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.histplot(dataframes[main_rs][numeric_field], bins=15, kde=True)
plt.title(f"Distribution of '{numeric_field}' in Record Set {main_rs}")
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# Boxplot by group if possible
if group_field and group_field in dataframes[main_rs].columns:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=group_field, y=numeric_field, data=dataframes[main_rs])
    plt.title(f"{numeric_field} by {group_field} in {main_rs}")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR\^2 clinical dataset schema enables programmatic exploration of clinical and molecular features using record set and field `@id`s.
- The main record set provides columns suitable for statistical and group-wise analysis.
- Numeric fields (e.g., age, intervals, or counts) can be filtered and normalized, supporting outcome modeling or subgroup analysis.
- Group-level comparisons (by comorbidity, anatomical site, etc.) are possible using the standardized fields.
- Further analysis may include modeling relationships between molecular markers and clinical variables as supported by the record set structure.